# AirIntel – Notebook 13: Streamlit Dashboard Architecture & Web Application

**Objective**: Document the production architecture, single-page conditional execution routing, backend integration, UI/UX design system, verification suite, and deployment workflow for the AirIntel Streamlit web application (`app.py`).

Notebook 13 serves as the engineering specification and system verification guide. The interactive application logic resides inside `app.py` and modular components within the `components/` directory.

## 01. Introduction

AirIntel features a single-page conditional execution routing architecture in `app.py`. Unlike default multi-tab frameworks that compute all tab contents on every rerun, AirIntel evaluates ONLY the active page selected via `st.segmented_control()`. This eliminates duplicate computations, prevents repeated sidebar rebuilds, and minimizes load times.

## 02. Project Overview

Summary of the end-to-end AirIntel pipeline status up to Notebook 13.

In [1]:
# Project environment setup and verification
import sys
import os
import json
import pickle
from pathlib import Path
import pandas as pd
import numpy as np

PROJECT_DIR = Path(r"c:\Users\kayri\OneDrive - IIT BHU\Documents\Indian_Air_Quality_Project")
DEPLOYMENT_DIR = PROJECT_DIR / "models" / "deployment"
BUNDLE_PATH = DEPLOYMENT_DIR / "deployment_pipeline.pkl"

print(f"Project Directory: {PROJECT_DIR}")
print(f"Deployment Bundle Path Exists: {BUNDLE_PATH.exists()}")

Project Directory: c:\Users\kayri\OneDrive - IIT BHU\Documents\Indian_Air_Quality_Project
Deployment Bundle Path Exists: True


## 03. Single-Page Routing Architecture

The application follows strict single-page conditional routing:

```text
1. Page Configuration (st.set_page_config)
2. Load CSS (with open('assets/style.css'))
3. Load Deployment Bundle (deployment_pipeline.pkl)
4. Import Components (components/*.py)
5. Render ONE Navigation Bar (st.segmented_control)
6. Render ONE Sidebar (render_sidebar)
7. Render ONE Page (Conditional routing: if page == 'X': render_X())
8. Footer (Rendered once)
```

## 04. Application Structure

The application is organized into modular components for maintainability:

• `app.py`: Main Streamlit entrypoint with `st.segmented_control` page selector and single-pass conditional routing.
• `assets/style.css`: Custom v2 light-theme SaaS styling (`#F8FAFC`), rounded card panels (`16px`), soft shadows, and status badges.
• `components/utils.py`: Cached pipeline loader (`load_deployment_bundle`), ISO timestamp generator, and system health evaluator.
• `components/sidebar.py`: Contextual dynamic sidebar panel adapting controls matching active page.
• `components/overview.py`: Home landing page with Hero header, KPI cards, visual workflow, project highlights, and scientific discoveries.
• `components/analytics.py`: Tableau/PowerBI-style Analytics page with 7 key visualizations (Title -> Chart -> One-line insight).
• `components/prediction.py`: 2-Column interactive prediction module (AQI gauge chart, SHAP drivers, recommendations, history).
• `components/maps.py`: Pydeck spatial analytics map for 29 Indian cities with regional contrast cards.
• `components/diagnostics.py`: Explainability page (TreeSHAP attributions), Architecture page (topology flow), and About page (3x3 grid cards).
• `components/reports.py`: Downloads page with 4 cards and JSON payload preview.

## 05. Backend Integration

Load and verify the serialized production bundle from Notebook 12.

In [2]:
# Load deployment pipeline bundle dynamically
with open(BUNDLE_PATH, "rb") as f:
    pipeline_bundle = pickle.load(f)

print(f"Loaded Engine: {pipeline_bundle.get('engine', 'AirIntel v2.0')}")
print(f"Model Version: {pipeline_bundle.get('model_version', 'v2.0')}")
print(f"Valid Cities Count: {len(pipeline_bundle.get('valid_cities', []))}")
print(f"Selected Features Count: {len(pipeline_bundle.get('selected_features', []))}")

C:\Users\kayri\AppData\Local\uv\cache\archive-v0\WsSHkWK0_23eA664M_Gw2\Lib\site-packages\sklearn\base.py:525: InconsistentVersionWarning: Trying to unpickle estimator SimpleImputer from version 1.7.2 when using version 1.9.0. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(
C:\Users\kayri\AppData\Local\uv\cache\archive-v0\WsSHkWK0_23eA664M_Gw2\Lib\site-packages\sklearn\base.py:525: InconsistentVersionWarning: Trying to unpickle estimator StandardScaler from version 1.7.2 when using version 1.9.0. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(
C:\Users\kayri\AppData\Local\uv\cache\archive-v0\WsSHkWK0_23eA664M_Gw2\Lib\site-packages\sklearn\base.py:525: Inconsi

Loaded Engine: AirIntel v1.0
Model Version: v1.0
Valid Cities Count: 29
Selected Features Count: 36


C:\Users\kayri\AppData\Local\uv\cache\archive-v0\WsSHkWK0_23eA664M_Gw2\Lib\site-packages\sklearn\base.py:525: InconsistentVersionWarning: Trying to unpickle estimator LabelEncoder from version 1.7.2 when using version 1.9.0. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(


## 06. Scientific Workflow Verification

Verify Scientific Mode inference using measured pollutant inputs.

In [3]:
# Scientific Mode pipeline verification test
reg_pipeline = pipeline_bundle['reg_pipeline']
cls_pipeline = pipeline_bundle['cls_pipeline']
label_encoder = pipeline_bundle['label_encoder']
selected_features = pipeline_bundle['selected_features']
feature_medians = pipeline_bundle['feature_medians']

sample_input = {col: feature_medians.get(col, 0.0) for col in selected_features}
sample_input['City'] = 'Delhi'
sample_input['Temp_2m_C'] = 35.0
sample_input['PM2.5'] = 120.0

df_sample = pd.DataFrame([sample_input])
cat_cols = ['City', 'Season', 'Wind_Category', 'Latitude_Band', 'Longitude_Band', 'Time_of_Day', 'Humidity_Category']
for col in cat_cols:
    if col in df_sample.columns:
        df_sample[col] = df_sample[col].astype(str)

df_aligned = df_sample[selected_features]

aqi_val = float(reg_pipeline.predict(df_aligned)[0])
probs = cls_pipeline.predict_proba(df_aligned)[0]
cat_name = label_encoder.inverse_transform([np.argmax(probs)])[0]

print(f"Scientific Mode Output -> Predicted AQI: {aqi_val:.2f}, Category: {cat_name}")

Scientific Mode Output -> Predicted AQI: 100.98, Category: Unhealthy_Sensitive


## 07. Public Workflow Verification

Verify Public Mode input validation and intentional deferred response contract.

In [4]:
# Public Mode pipeline verification test
public_payload = {
    "City": "Mumbai",
    "Temp_2m_C": 28.5,
    "Humidity_Percent": 65.0,
    "Month": 7
}

response_contract = {
    "Engine_Status": "Deployment Ready",
    "Prediction_Status": "Awaiting Forecast Model",
    "Reason": "This interface prepares validated weather and location features for a future weather-based AQI forecasting model.",
    "Input_City": public_payload["City"],
    "Prediction_Mode": "public"
}

print("Public Mode Contract:", json.dumps(response_contract, indent=2))

Public Mode Contract: {
  "Engine_Status": "Deployment Ready",
  "Prediction_Status": "Awaiting Forecast Model",
  "Reason": "This interface prepares validated weather and location features for a future weather-based AQI forecasting model.",
  "Input_City": "Mumbai",
  "Prediction_Mode": "public"
}


## 08. UI Components Breakdown

The application relies on modular Python files inside `components/`:

• `utils.py`: Cached loader (`load_deployment_bundle`), ISO timestamp generator, and `get_system_health_status` evaluator.
• `sidebar.py`: Contextual dynamic sidebar panel adapting controls matching active page.
• `overview.py`: Home landing page with Hero header, KPI cards, visual workflow, project highlights, and scientific discoveries.
• `analytics.py`: EDA & Data Insights page with 7 key visualizations (Title -> Chart -> One-line insight).
• `prediction.py`: 2-Column interactive prediction module (AQI gauge chart, SHAP drivers, recommendations, history).
• `maps.py`: Pydeck spatial analytics map for 29 Indian cities with regional contrast cards.
• `diagnostics.py`: Explainability page (TreeSHAP attributions), Architecture page (topology flow), and About page (3x3 grid cards).
• `reports.py`: Downloads page with 4 cards and JSON payload preview.

## 09. System Testing Suite

Verify component compilation and entrypoint integrity.

In [5]:
# Verify component module imports
import py_compile
import glob

modules = glob.glob(str(PROJECT_DIR / "components" / "*.py")) + [str(PROJECT_DIR / "app.py")]
for mod in modules:
    py_compile.compile(mod, doraise=True)
print(f"Successfully compiled {len(modules)} application Python files.")

Successfully compiled 10 application Python files.


## 10. Summary

### Key Accomplishments in Notebook 13 & app.py

1. **Single-Page Conditional Routing**: Refactored `app.py` to evaluate only the selected page via `st.segmented_control()`.
2. **Single Sidebar Render**: Rendered sidebar exactly once per rerun using `sidebar_output, prediction_mode = render_sidebar(selected_page, ...)`.
3. **Zero Duplicate Computation**: Non-selected pages do not execute, preventing redundant SHAP or Pydeck map calculations.
4. **Zero Backend Changes**: Preserved 100% backend compatibility with Notebook 12 deployment bundle without touching models or inference logic.